# 02 — Distillation

SS9 Phase 5. Teacher generation with vLLM, then the three KD arms.

Cross-tokenizer note: Qwen vocab is ~151k, ours is 16k, so logit KD is impossible
across them. Arms 1-2 are tokenizer-agnostic; arm 3 uses our own 100M sibling.


In [ ]:
# Thin launcher: clone, install, run. No project logic lives in this notebook.
import subprocess, sys, os, pathlib
REPO = 'https://github.com/adnanbasil10/localmind.git'
WORK = pathlib.Path('/kaggle/working/localmind')
if not WORK.exists():
    subprocess.run(['git','clone','--depth','1',REPO,str(WORK)], check=True)
os.chdir(WORK)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.[torch,tok,data]'], check=True)
import torch; print('torch',torch.__version__,'| cuda',torch.cuda.is_available(),
      '|', torch.cuda.device_count(),'x', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


In [ ]:
# T4 is SM 7.5: fp16 + GradScaler only. Fail loudly if someone puts bf16 in a config.
import torch
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    print('compute capability', cap)
    assert not torch.cuda.is_bf16_supported() or cap[0] >= 8, 'bf16 claim on pre-Ampere'
    print('bf16 supported:', torch.cuda.is_bf16_supported(), '-> plan mandates fp16 regardless')


In [ ]:
# Session-cap insurance (SS3.2 item 3): push checkpoints to HF Hub every hour.
# The HF username is derived from your token, so there is nothing to edit here.
import os
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')

from huggingface_hub import whoami
user = whoami(token=os.environ['HF_TOKEN'])['name']
os.environ['LOCALMIND_HUB_REPO'] = f'{user}/localmind-31m'
print('checkpoints will push to:', os.environ['LOCALMIND_HUB_REPO'])


## Teacher generation (Qwen2.5-3B-Instruct via vLLM, ~1-2 h for 50k short outputs)


In [ ]:
!pip install -q vllm
!python -m localmind.post.sft --generate-teacher-data \n    --teacher Qwen/Qwen2.5-3B-Instruct --n 50000 --out /kaggle/working/sft_data.jsonl


## Arm 1 — sequence-level KD (primary, tokenizer-agnostic)


In [ ]:
!python -m localmind.post.kd --config configs/train/kd.yaml --arm sequence


## Arm 2 — on-policy correction (fixes exposure bias)


In [ ]:
!python -m localmind.post.kd --config configs/train/kd.yaml --arm on-policy


## Arm 3 — same-tokenizer logit KD from LocalMind-100M (optional, ~10 extra GPU-h)
Top-64 logits over 20k x 256 tokens is ~1.2 GB — dump to Hub, don't hold in session.


In [ ]:
!python -m localmind.post.kd --config configs/train/kd.yaml --arm logit --top-k 64 --alpha 0.7


## DPO then GRPO


In [ ]:
!python -m localmind.post.dpo --config configs/train/dpo.yaml --beta 0.1
!python -m localmind.post.grpo --config configs/train/grpo.yaml --group-size 8
